<h1> Cost-Aware Ensemble Intelligence for Bankruptcy Prediction </h1>

In [1]:
# 1: Importing libraries and setting configuration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb
import lightgbm as lgb

from sklearn.exceptions import ConvergenceWarning

# Suppressing convergence warnings during model fitting
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Setting global random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


In [2]:
# 2: Loading dataset and renaming long column names

# Loading dataset
try:
    df = pd.read_csv("company_bankruptcy_dataset.csv")
except FileNotFoundError:
    raise SystemExit("Error: 'company_bankruptcy_dataset.csv' not found in the current directory.")

# Renaming long financial columns to compact names where applicable
rename_mapping = {
    'Current assets: All the assets of a company that are expected to be sold or used as a result of standard business operations over the next year\n': 'X1',
    'Cost of goods sold: The total amount a company paid as a cost directly related to the sale of products': 'X2',
    'Depreciation and amortization: Depreciation refers to the loss of value of a tangible fixed asset over time (such as property, machinery, buildings, and plant). Amortization refers to the loss of value of intangible assets over time.': 'X3',
    'EBITDA: Earnings before interest, taxes, depreciation, and amortization. A measure of a company’s overall financial performance alternative to net income': 'X4',
    'Inventory: The accounting of items and raw materials that a company either uses in production or sells': 'X5',
    'Net Income: The overall profitability of a company after all expenses and costs have been deducted from total revenue': 'X6',
    'Total Receivables: The balance of money due to a firm for goods or services delivered or used but not yet paid for by customers': 'X7',
    'Market Value: The price of an asset in a marketplace. In our dataset, it refers to the market capitalization since companies are publicly traded in the stock market': 'X8',
    'Net Sales: The sum of a company’s gross sales minus its returns, allowances, and discounts': 'X9',
    'Total Assets: All the assets, or items of value, a business owns': 'X10',
    'Total Long-term Debt: A company’s loans and other liabilities that will not become due within one year of the balance sheet date': 'X11',
    'EBIT: Earnings before interest and taxes': 'X12',
    'Gross Profit: The profit a company makes after deducting the costs associated with making and selling its products': 'X13',
    'Total Current Liabilities: A company’s debts or obligations that are due within one year': 'X14',
    'Retained Earnings: The accumulated net income of a corporation that is retained by the corporation at the end of each reporting period': 'X15',
    'Total Revenue: The total income generated by the sale of goods or services related to the company’s primary operations': 'X16',
    'Total Short-term Debt: Debts or obligations that must be paid within one year or one operating cycle, whichever is longer': 'X17',
    'Cash flow from operating activities: The amount of cash generated by the regular operating activities of a business within a specific time frame': 'X18'
}
df = df.rename(columns={k: v for k, v in rename_mapping.items() if k in df.columns})

print("Initial data shape:", df.shape)
print(df.head())


Initial data shape: (78682, 21)
  company_name status_label  year       X1       X2      X3      X4       X5  \
0          C_1        alive  1999  511.267  833.107  18.373  89.031  336.018   
1          C_1        alive  2000  485.856  713.811  18.577  64.367  320.590   
2          C_1        alive  2001  436.656  526.477  22.496  27.207  286.588   
3          C_1        alive  2002  396.412  496.747  27.172  30.745  259.954   
4          C_1        alive  2003  432.204  523.302  26.680  47.491  247.245   

       X6       X7  ...        X9      X10      X11     X12  \
0  35.163  128.348  ...  1024.333  740.998  180.447  70.658   
1  18.531  115.187  ...   874.255  701.854  179.987  45.790   
2 -58.939   77.528  ...   638.721  710.199  217.699   4.711   
3 -12.410   66.322  ...   606.337  686.621  164.658   3.573   
4   3.504  104.661  ...   651.958  709.292  248.666  20.811   

   Gross Profit: The profit a business makes after subtracting all the costs that are related to manufacturi

In [3]:
# 3: Cleaning columns, dropping missing values, encoding target

# Dropping unnamed index-like columns if present
unnamed_cols = [c for c in df.columns if "unnamed" in c.lower()]
df = df.drop(columns=unnamed_cols, errors="ignore")

print("\nData info before dropping NAs:")
print(df.info())

# Handling missing values by dropping incomplete rows
df = df.dropna(axis=0)
print("\nData shape after dropping missing values:", df.shape)

# Encoding bankruptcy status: 1 = failed/bankrupt, 0 = alive
if df["status_label"].dtype == "O":
    df["status_label"] = df["status_label"].map({"failed": 1, "alive": 0})

print("\nTarget value counts after encoding:")
print(df["status_label"].value_counts())



Data info before dropping NAs:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78682 entries, 0 to 78681
Data columns (total 21 columns):
 #   Column                                                                                                                                                                    Non-Null Count  Dtype  
---  ------                                                                                                                                                                    --------------  -----  
 0   company_name                                                                                                                                                              78682 non-null  object 
 1   status_label                                                                                                                                                              78682 non-null  object 
 2   year                                                      

In [4]:
# 4: Detecting year column and preparing features and target

# Identifying year column for temporal evaluation if available
year_col = None
for candidate in ["year", "Year", "fyear", "fiscal_year"]:
    if candidate in df.columns:
        year_col = candidate
        break

if year_col is not None:
    print(f"\nUsing '{year_col}' as temporal column for rolling evaluation.")
else:
    print("\nNo explicit year column found. Temporal rolling evaluation will be skipped.")

# Separating features and target
non_feature_cols = ["company_name", "status_label"]
if year_col is not None:
    non_feature_cols.append(year_col)

feature_cols = [c for c in df.columns if c not in non_feature_cols]
X = df[feature_cols].copy()
y = df["status_label"].copy()

print("\nFeature columns used:")
print(feature_cols)



Using 'year' as temporal column for rolling evaluation.

Feature columns used:
['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'Gross Profit: The profit a business makes after subtracting all the costs that are related to manufacturing and selling its products or services', 'Total Current Liabilities: The sum of accounts payable, accrued liabilities, and taxes such as bonds payable at the end of the year, salaries, and commissions remaining', 'Retained Earnings: The amount of profit a company has left over after paying all its direct costs, indirect costs, income taxes, and dividends to shareholders', 'Total Revenue: The amount of income that a business has made from all sales before subtracting expenses. It may include interest and dividends from investments', 'Total Liabilities: The combined debts and obligations that the company owes to outside parties', 'Total Operating Expenses: The expense a business incurs through its normal business operations']


In [5]:
# 5: Plotting class distribution and creating train-test split

# Plotting class distribution
class_counts = y.value_counts().sort_index()
print("\nClass distribution (0 = alive, 1 = failed):")
print(class_counts)

plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title("Class distribution of bankruptcy status")
plt.xlabel("Status (0 = alive, 1 = failed)")
plt.ylabel("Number of companies")
plt.tight_layout()
plt.savefig("class_distribution.png")
plt.close()

# Performing train-test split with stratification
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("\nTrain shape (full):", X_train_full.shape)
print("Test shape:", X_test.shape)

# Defining cross-validation strategy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)



Class distribution (0 = alive, 1 = failed):
status_label
0    73462
1     5220
Name: count, dtype: int64

Train shape (full): (62945, 18)
Test shape: (15737, 18)


In [6]:
# 6: Performing feature selection using Random Forest

rf_fs = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf_fs.fit(X_train_full, y_train_full)
importances = pd.Series(rf_fs.feature_importances_, index=feature_cols).sort_values(ascending=False)

top_k = 10
selected_features = importances.head(top_k).index.tolist()
print(f"\nTop {top_k} selected features based on Random Forest importance:")
print(selected_features)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.head(top_k).values, y=importances.head(top_k).index)
plt.title(f"Top {top_k} features by Random Forest importance")
plt.xlabel("Feature importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("rf_feature_importance_top10.png")
plt.close()

# Restricting data to selected features for main models
X_train = X_train_full[selected_features].copy()
X_test_sel = X_test[selected_features].copy()
y_train = y_train_full.loc[X_train.index].copy()



Top 10 selected features based on Random Forest importance:
['X8', 'X6', 'Retained Earnings: The amount of profit a company has left over after paying all its direct costs, indirect costs, income taxes, and dividends to shareholders', 'X3', 'X1', 'Total Liabilities: The combined debts and obligations that the company owes to outside parties', 'X7', 'Gross Profit: The profit a business makes after subtracting all the costs that are related to manufacturing and selling its products or services', 'X11', 'X2']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_28248\3871913715.py:23: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [7]:
# 7: Defining model evaluation helper function

def evaluate_model(model_name, estimator, X_train, y_train, X_test, y_test, cv,
                   metrics_list, proba_store, fitted_store):
    """
    Evaluating a model on the test set and storing key metrics.
    """
    model = clone(estimator)
    model.fit(X_train, y_train)

    # Predicting labels and probabilities
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        if hasattr(model, "decision_function"):
            scores = model.decision_function(X_test)
            y_proba = 1 / (1 + np.exp(-scores))
        else:
            y_proba = None

    # Computing metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    if y_proba is not None:
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    # Performing cross-validation using ROC-AUC where possible
    try:
        cv_score = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    except Exception as e:
        print(f"Warning: CV failed for {model_name} with error: {e}")
        cv_score = np.nan

    metrics_list.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "CV_ROC_AUC": cv_score
    })

    proba_store[model_name] = y_proba
    fitted_store[model_name] = model

    # Printing evaluation report
    print(f"\n--- {model_name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    if not np.isnan(roc_auc):
        print(f"ROC-AUC: {roc_auc:.4f}")
        print(f"PR-AUC: {pr_auc:.4f}")
    else:
        print("ROC-AUC / PR-AUC: Not available")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))


In [8]:
# 8: Defining hyperparameter tuning helper function

def tune_model(estimator, param_distributions, X, y, cv, n_iter=20, scoring="roc_auc", label="model"):
    """
    Running randomized search for tuning and returning the best estimator.
    Uses NumPy arrays to avoid issues with feature names (e.g., LightGBM JSON restrictions).
    """
    # Converting to NumPy arrays to avoid LightGBM feature-name errors
    if isinstance(X, pd.DataFrame):
        X_in = X.values
    else:
        X_in = X

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring=scoring,
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_in, y)
    print(f"\nBest {label} params:", search.best_params_)
    print(f"Best {label} ROC-AUC (CV): {search.best_score_:.4f}")
    return search.best_estimator_


In [9]:
# 9: Defining baseline models for all algorithms

baseline_models = {}

# Logistic Regression
baseline_models["Logistic Regression"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=500,
        class_weight="balanced"
    ))
])

# Random Forest (baseline)
baseline_models["Random Forest (baseline)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

# SVM (Linear)
baseline_models["SVM (Linear)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="linear",
        probability=True,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

# Decision Tree
baseline_models["Decision Tree"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

# KNN
baseline_models["K-Nearest Neighbors"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(
        n_neighbors=7,
        weights="distance"
    ))
])

# Naive Bayes
baseline_models["Naive Bayes"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GaussianNB())
])

# Gradient Boosting
baseline_models["Gradient Boosting"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ))
])

# XGBoost (baseline)
baseline_models["XGBoost (baseline)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# LightGBM (baseline)
baseline_models["LightGBM (baseline)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ))
])

print("\nBaseline models defined.")



Baseline models defined.


In [10]:
# 10: Tuning Random Forest

print("\nTuning Random Forest...")

rf_clf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf_param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [None, 4, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}
rf_tuned = tune_model(
    estimator=rf_clf,
    param_distributions=rf_param_dist,
    X=X_train,
    y=y_train,
    cv=skf,
    n_iter=20,
    label="Random Forest"
)



Tuning Random Forest...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Random Forest params: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best Random Forest ROC-AUC (CV): 0.8510


In [11]:
# 11: Tuning XGBoost

print("\nTuning XGBoost...")

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_param_dist = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 6],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "gamma": [0, 0.1, 0.2]
}
xgb_tuned = tune_model(
    estimator=xgb_clf,
    param_distributions=xgb_param_dist,
    X=X_train,
    y=y_train,
    cv=skf,
    n_iter=20,
    label="XGBoost"
)



Tuning XGBoost...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best XGBoost params: {'subsample': 0.8, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.7}
Best XGBoost ROC-AUC (CV): 0.8162


In [12]:
# 12: Tuning LightGBM and defining tuned models + stacking ensemble

print("\nTuning LightGBM...")

lgb_clf = lgb.LGBMClassifier(
    objective="binary",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)
lgb_param_dist = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [-1, 4, 6, 8],
    "num_leaves": [31, 63, 127],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}
lgb_tuned = tune_model(
    estimator=lgb_clf,
    param_distributions=lgb_param_dist,
    X=X_train,
    y=y_train,
    cv=skf,
    n_iter=20,
    label="LightGBM"
)

# Defining enhanced tuned models with scaling
models = {}

models["Logistic Regression"] = baseline_models["Logistic Regression"]

models["Random Forest (tuned)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", rf_tuned)
])

models["SVM (Linear)"] = baseline_models["SVM (Linear)"]
models["Decision Tree"] = baseline_models["Decision Tree"]
models["K-Nearest Neighbors"] = baseline_models["K-Nearest Neighbors"]
models["Naive Bayes"] = baseline_models["Naive Bayes"]
models["Gradient Boosting"] = baseline_models["Gradient Boosting"]

models["XGBoost (tuned)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", xgb_tuned)
])

models["LightGBM (tuned)"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", lgb_tuned)
])

# Defining champion RF with SMOTE and tuned hyperparameters
champion_name = "Random Forest (Tuned + SMOTE)"
champion_model = ImbPipeline([
    ("smote", SMOTE(
        sampling_strategy="auto",
        random_state=RANDOM_STATE,
        k_neighbors=5
    )),
    ("scaler", StandardScaler()),
    ("clf", clone(rf_tuned))
])
models[champion_name] = champion_model

# Defining stacking ensemble
stack_base_estimators = [
    ("rf", models["Random Forest (tuned)"]),
    ("xgb", models["XGBoost (tuned)"]),
    ("lgb", models["LightGBM (tuned)"]),
    ("logreg", models["Logistic Regression"]),
    ("svm", models["SVM (Linear)"]),
    ("gb", models["Gradient Boosting"])
]

stack_final_estimator = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=500,
    class_weight="balanced"
)

stacking_model = StackingClassifier(
    estimators=stack_base_estimators,
    final_estimator=stack_final_estimator,
    cv=skf,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

stacking_name = "Stacking Ensemble (All tuned)"
models[stacking_name] = stacking_model

print("\nTuned models and stacking ensemble defined.")



Tuning LightGBM...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Number of positive: 4176, number of negative: 58769
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003681 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 62945, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000

Best LightGBM params: {'subsample': 0.8, 'num_leaves': 127, 'n_estimators': 400, 'max_depth': -1, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
Best LightGBM ROC-AUC (CV): 0.8304

Tuned models and stacking ensemble defined.


In [13]:
# 13: Evaluating all models and generating comparison plots

metrics_list = []
proba_store = {}
fitted_store = {}

for name, est in models.items():
    evaluate_model(
        model_name=name,
        estimator=est,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test_sel,
        y_test=y_test,
        cv=skf,
        metrics_list=metrics_list,
        proba_store=proba_store,
        fitted_store=fitted_store
    )

metrics_df = pd.DataFrame(metrics_list).sort_values(by="ROC_AUC", ascending=False)
print("\n=== Model comparison summary ===")
print(metrics_df)

metrics_df.to_csv("model_comparison_summary_enhanced.csv", index=False)

# Plotting metric comparisons
plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x="Model", y="Accuracy")
plt.xticks(rotation=45, ha="right")
plt.title("Accuracy comparison across models")
plt.tight_layout()
plt.savefig("accuracy_comparison_models.png")
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x="Model", y="ROC_AUC")
plt.xticks(rotation=45, ha="right")
plt.title("ROC-AUC comparison across models")
plt.tight_layout()
plt.savefig("roc_auc_comparison_models.png")
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x="Model", y="PR_AUC")
plt.xticks(rotation=45, ha="right")
plt.title("PR-AUC comparison across models")
plt.tight_layout()
plt.savefig("pr_auc_comparison_models.png")
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x="Model", y="F1")
plt.xticks(rotation=45, ha="right")
plt.title("F1 score comparison across models")
plt.tight_layout()
plt.savefig("f1_comparison_models.png")
plt.close()

# Plotting ROC and PR curves for top 3 models by ROC-AUC
top_models = metrics_df.head(3)["Model"].tolist()

plt.figure(figsize=(7, 6))
for name in top_models:
    y_proba = proba_store[name]
    if y_proba is None:
        continue
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves for top models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curves_top_models.png")
plt.close()

plt.figure(figsize=(7, 6))
for name in top_models:
    y_proba = proba_store[name]
    if y_proba is None:
        continue
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    plt.plot(recall, precision, label=f"{name} (AP = {pr_auc:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves for top models")
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig("pr_curves_top_models.png")
plt.close()



--- Logistic Regression ---
Accuracy: 0.3573
F1 Score: 0.1518
ROC-AUC: 0.6752
PR-AUC: 0.1393

Confusion Matrix:
[[4718 9975]
 [ 139  905]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9714    0.3211    0.4827     14693
           1     0.0832    0.8669    0.1518      1044

    accuracy                         0.3573     15737
   macro avg     0.5273    0.5940    0.3172     15737
weighted avg     0.9125    0.3573    0.4607     15737


--- Random Forest (tuned) ---
Accuracy: 0.9377
F1 Score: 0.1249
ROC-AUC: 0.8676
PR-AUC: 0.4566

Confusion Matrix:
[[14686     7]
 [  974    70]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9378    0.9995    0.9677     14693
           1     0.9091    0.0670    0.1249      1044

    accuracy                         0.9377     15737
   macro avg     0.9234    0.5333    0.5463     15737
weighted avg     0.9359    0.9377    0.9118     15737


--- SVM (Li

In [14]:
# 14: Champion analysis - calibration, thresholds, risk index, scenarios, permutation importance

# Selecting final champion: prefer stacking if available
if stacking_name in fitted_store:
    final_champion_name = stacking_name
else:
    final_champion_name = champion_name

print(f"\nUsing '{final_champion_name}' as final champion model for advanced analysis.")

champion_fitted = fitted_store[final_champion_name]
champion_proba_test = proba_store[final_champion_name]

# Plotting calibration curve and computing Brier score
prob_true, prob_pred = calibration_curve(y_test, champion_proba_test, n_bins=10, strategy="quantile")
brier = brier_score_loss(y_test, champion_proba_test)
print(f"\nChampion model Brier score (lower is better): {brier:.4f}")

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", label="Champion")
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration curve - Champion model")
plt.legend()
plt.tight_layout()
plt.savefig("calibration_curve_champion.png")
plt.close()

# Performing threshold sweep with cost-based analysis
thresholds = np.linspace(0.1, 0.9, 17)
threshold_metrics = []

cost_fp = 1.0   # Cost of investigating a healthy firm
cost_fn = 5.0   # Cost of missing a bankrupt firm

for thresh in thresholds:
    y_pred_thresh = (champion_proba_test >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_thresh, labels=[0, 1]).ravel()
    acc = accuracy_score(y_test, y_pred_thresh)
    f1_val = f1_score(y_test, y_pred_thresh)
    precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    total_cost = cost_fn * fn + cost_fp * fp
    avg_cost = total_cost / len(y_test)

    threshold_metrics.append({
        "threshold": thresh,
        "accuracy": acc,
        "f1": f1_val,
        "precision": precision_val,
        "recall": recall_val,
        "avg_cost": avg_cost
    })

threshold_df = pd.DataFrame(threshold_metrics)
best_row = threshold_df.loc[threshold_df["avg_cost"].idxmin()]

print("\n=== Threshold analysis for champion model ===")
print(threshold_df)
print("\nBest threshold by average cost:")
print(best_row)

threshold_df.to_csv("champion_threshold_analysis.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1 Score")
plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.title("Threshold vs Precision/Recall/F1 (Champion model)")
plt.legend()
plt.tight_layout()
plt.savefig("threshold_vs_metrics_champion.png")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["avg_cost"], marker="o")
plt.xlabel("Decision threshold")
plt.ylabel("Average cost per company")
plt.title("Threshold vs expected cost (Champion model)")
plt.tight_layout()
plt.savefig("threshold_vs_cost_champion.png")
plt.close()

# Constructing leverage- and size-aware risk index
eps = 1e-6
test_indices = X_test_sel.index
df_test = df.loc[test_indices].copy()

if all(col in df_test.columns for col in ["X10", "X11", "X17", "X8"]):
    total_assets = df_test["X10"].astype(float) + eps
    long_debt = df_test["X11"].astype(float)
    short_debt = df_test["X17"].astype(float)
    market_value = df_test["X8"].astype(float) + eps

    leverage_ratio = (long_debt + short_debt) / total_assets
    size_factor = np.log1p(market_value)

    risk_index = champion_proba_test * (1 + leverage_ratio) / (1 + size_factor / size_factor.median())
    df_test["risk_index"] = risk_index
    df_test["pred_proba_champion"] = champion_proba_test
    df_test["y_true"] = y_test.values

    # Analyzing risk deciles
    df_test["risk_decile"] = pd.qcut(df_test["risk_index"], 10, labels=False, duplicates="drop")
    decile_summary = df_test.groupby("risk_decile").agg(
        n_companies=("y_true", "size"),
        n_failed=("y_true", "sum"),
        avg_risk_index=("risk_index", "mean")
    )
    decile_summary["failure_rate"] = decile_summary["n_failed"] / decile_summary["n_companies"]
    print("\n=== Risk index decile analysis ===")
    print(decile_summary.sort_index(ascending=False))

    decile_summary.sort_index(inplace=True)
    plt.figure(figsize=(8, 5))
    plt.bar(decile_summary.index.astype(str), decile_summary["failure_rate"])
    plt.xlabel("Risk decile (0 = lowest risk, 9 = highest risk)")
    plt.ylabel("Observed bankruptcy rate")
    plt.title("Observed bankruptcy rate by risk decile (Champion risk index)")
    plt.tight_layout()
    plt.savefig("risk_decile_failure_rates.png")
    plt.close()
else:
    print("\nInsufficient columns to compute leverage- and size-aware risk index (need X8, X10, X11, X17).")

# Performing scenario analysis using top-risk screening
if "risk_index" in df_test.columns:
    df_test_sorted = df_test.sort_values(by="risk_index", ascending=False)
    n_total = df_test_sorted.shape[0]
    n_failed_total = df_test_sorted["y_true"].sum()

    screening_levels = [0.01, 0.02, 0.05, 0.10, 0.20]
    scenario_records = []

    for level in screening_levels:
        k = max(1, int(n_total * level))
        top_k = df_test_sorted.head(k)
        captured_failed = top_k["y_true"].sum()
        capture_rate = captured_failed / n_failed_total if n_failed_total > 0 else 0.0
        flagged_share = k / n_total

        scenario_records.append({
            "screening_level": level,
            "n_flagged": k,
            "flagged_share": flagged_share,
            "captured_failed": captured_failed,
            "capture_rate": capture_rate
        })

    scenario_df = pd.DataFrame(scenario_records)
    print("\n=== Scenario analysis: screening by risk index ===")
    print(scenario_df)
    scenario_df.to_csv("risk_screening_scenarios.csv", index=False)

    plt.figure(figsize=(7, 5))
    plt.plot(scenario_df["flagged_share"] * 100, scenario_df["capture_rate"] * 100, marker="o")
    plt.xlabel("Percentage of firms flagged for review (%)")
    plt.ylabel("Percentage of bankruptcies captured (%)")
    plt.title("Screening efficiency curve (Champion risk index)")
    plt.tight_layout()
    plt.savefig("screening_efficiency_curve.png")
    plt.close()

# Performing permutation importance for champion model
try:
    perm_result = permutation_importance(
        champion_fitted,
        X_test_sel,
        y_test,
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    perm_importances = pd.Series(perm_result.importances_mean, index=selected_features).sort_values(ascending=False)
    print("\n=== Permutation feature importance (Champion model) ===")
    print(perm_importances)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=perm_importances.values, y=perm_importances.index)
    plt.title("Permutation feature importance - Champion model")
    plt.xlabel("Mean importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig("permutation_importance_champion.png")
    plt.close()
except Exception as e:
    print(f"\nCould not compute permutation importance due to: {e}")



Using 'Stacking Ensemble (All tuned)' as final champion model for advanced analysis.

Champion model Brier score (lower is better): 0.1410

=== Threshold analysis for champion model ===
    threshold  accuracy        f1  precision    recall  avg_cost
0        0.10  0.111838  0.129972   0.069503  1.000000  0.888162
1        0.15  0.170935  0.137502   0.073848  0.996169  0.830082
2        0.20  0.371418  0.170691   0.093532  0.975096  0.635191
3        0.25  0.527419  0.208577   0.117323  0.938697  0.488848
4        0.30  0.630997  0.243388   0.140854  0.894636  0.396963
5        0.35  0.705535  0.276390   0.165112  0.847701  0.334880
6        0.40  0.759484  0.307917   0.190282  0.806513  0.291860
7        0.45  0.800153  0.339424   0.217380  0.773946  0.259834
8        0.50  0.831480  0.363724   0.242638  0.726054  0.241215
9        0.55  0.857343  0.391764   0.273139  0.692529  0.224249
10       0.60  0.876025  0.412172   0.300659  0.655172  0.215479
11       0.65  0.894580  0.435906

C:\Users\ASUS\AppData\Local\Temp\ipykernel_28248\2301429301.py:186: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [15]:
# 15: Temporal rolling-origin evaluation and ablation study

# Running rolling-origin temporal evaluation if year information exists
if year_col is not None:
    years = df.loc[X.index, year_col]
    unique_years = np.sort(years.unique())

    rolling_records = []
    for i in range(len(unique_years) - 1):
        train_years = unique_years[: i + 1]
        test_year = unique_years[i + 1]

        train_mask = years.isin(train_years)
        test_mask = years == test_year

        if train_mask.sum() < 50 or test_mask.sum() < 20:
            continue

        X_train_roll = X.loc[train_mask, selected_features]
        y_train_roll = y.loc[train_mask]
        X_test_roll = X.loc[test_mask, selected_features]
        y_test_roll = y.loc[test_mask]

        # Using champion_model structure for rolling evaluation
        clf = clone(champion_model)
        clf.fit(X_train_roll, y_train_roll)
        y_proba_roll = clf.predict_proba(X_test_roll)[:, 1]
        y_pred_roll = (y_proba_roll >= 0.5).astype(int)

        acc = accuracy_score(y_test_roll, y_pred_roll)
        f1_val = f1_score(y_test_roll, y_pred_roll)
        roc_auc_val = roc_auc_score(y_test_roll, y_proba_roll)
        pr_auc_val = average_precision_score(y_test_roll, y_proba_roll)

        rolling_records.append({
            "train_year_min": train_years.min(),
            "train_year_max": train_years.max(),
            "test_year": test_year,
            "n_train": train_mask.sum(),
            "n_test": test_mask.sum(),
            "Accuracy": acc,
            "F1": f1_val,
            "ROC_AUC": roc_auc_val,
            "PR_AUC": pr_auc_val
        })

    if rolling_records:
        rolling_df = pd.DataFrame(rolling_records)
        print("\n=== Rolling-origin temporal evaluation (Champion-like model) ===")
        print(rolling_df)
        rolling_df.to_csv("rolling_origin_evaluation.csv", index=False)

        plt.figure(figsize=(8, 5))
        plt.plot(rolling_df["test_year"], rolling_df["ROC_AUC"], marker="o", label="ROC-AUC")
        plt.plot(rolling_df["test_year"], rolling_df["PR_AUC"], marker="s", label="PR-AUC")
        plt.xlabel("Test year")
        plt.ylabel("Score")
        plt.title("Temporal stability of model performance")
        plt.legend()
        plt.tight_layout()
        plt.savefig("temporal_performance_model.png")
        plt.close()
    else:
        print("\nNot enough samples per year to perform meaningful rolling-origin evaluation.")
else:
    print("\nSkipping temporal rolling-origin evaluation due to missing year column.")

# Running ablation study to quantify impact of SMOTE, FS, and Stacking
def run_ablation_study(X_train_full, y_train_full, X_test_full, y_test_full, selected_features, stacking_model):
    """
    Running an ablation study to quantify the impact of SMOTE,
    feature selection, and stacking ensemble.
    """
    results = []

    # RF - all features, no SMOTE
    rf_base = RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )
    rf_base.fit(X_train_full, y_train_full)
    y_proba = rf_base.predict_proba(X_test_full)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    results.append({
        "Setting": "RF - all features, no SMOTE",
        "Accuracy": accuracy_score(y_test_full, y_pred),
        "F1": f1_score(y_test_full, y_pred),
        "ROC_AUC": roc_auc_score(y_test_full, y_proba),
        "PR_AUC": average_precision_score(y_test_full, y_proba)
    })

    # RF - all features + SMOTE
    rf_smote = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", clone(rf_base))
    ])
    rf_smote.fit(X_train_full, y_train_full)
    y_proba = rf_smote.predict_proba(X_test_full)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    results.append({
        "Setting": "RF - all features + SMOTE",
        "Accuracy": accuracy_score(y_test_full, y_pred),
        "F1": f1_score(y_test_full, y_pred),
        "ROC_AUC": roc_auc_score(y_test_full, y_proba),
        "PR_AUC": average_precision_score(y_test_full, y_proba)
    })

    # RF - FS + SMOTE
    rf_smote_fs = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", clone(rf_base))
    ])
    rf_smote_fs.fit(X_train_full[selected_features], y_train_full)
    y_proba = rf_smote_fs.predict_proba(X_test_full[selected_features])[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    results.append({
        "Setting": "RF - FS + SMOTE",
        "Accuracy": accuracy_score(y_test_full, y_pred),
        "F1": f1_score(y_test_full, y_pred),
        "ROC_AUC": roc_auc_score(y_test_full, y_proba),
        "PR_AUC": average_precision_score(y_test_full, y_proba)
    })

    # Stacking - tuned, FS
    stack = clone(stacking_model)
    stack.fit(X_train_full[selected_features], y_train_full)
    y_proba = stack.predict_proba(X_test_full[selected_features])[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    results.append({
        "Setting": "Stacking - tuned, FS",
        "Accuracy": accuracy_score(y_test_full, y_pred),
        "F1": f1_score(y_test_full, y_pred),
        "ROC_AUC": roc_auc_score(y_test_full, y_proba),
        "PR_AUC": average_precision_score(y_test_full, y_proba)
    })

    return pd.DataFrame(results)

ablation_df = run_ablation_study(
    X_train_full=X_train_full,
    y_train_full=y_train_full,
    X_test_full=X_test,
    y_test_full=y_test,
    selected_features=selected_features,
    stacking_model=stacking_model
)

print("\n=== Ablation study (impact of SMOTE, FS, and Stacking) ===")
print(ablation_df)
ablation_df.to_csv("ablation_study_results.csv", index=False)

plt.figure(figsize=(8, 5))
for metric in ["ROC_AUC", "PR_AUC"]:
    plt.plot(ablation_df["Setting"], ablation_df[metric], marker="o", label=metric)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Score")
plt.title("Ablation study metrics")
plt.legend()
plt.tight_layout()
plt.savefig("ablation_study_metrics.png")
plt.close()

print("\nAdvanced pipeline completed successfully.")



=== Rolling-origin temporal evaluation (Champion-like model) ===
    train_year_min  train_year_max  test_year  n_train  n_test  Accuracy  \
0             1999            1999       2000     5308    5226  0.905090   
1             1999            2000       2001    10534    4897  0.897488   
2             1999            2001       2002    15431    4651  0.902817   
3             1999            2002       2003    20082    4417  0.901743   
4             1999            2003       2004    24499    4348  0.899264   
5             1999            2004       2005    28847    4205  0.897265   
6             1999            2005       2006    33052    4128  0.898498   
7             1999            2006       2007    37180    4009  0.902968   
8             1999            2007       2008    41189    3857  0.890848   
9             1999            2008       2009    45046    3743  0.911034   
10            1999            2009       2010    48789    3625  0.921103   
11            1999    